# Audio frontend

The input side of the network: waveform to the 80 channel log Mel
spectrogram, exactly as the runtime computes it (25 ms window, 10 ms hop,
FFT 400, hop 160, 30 second chunks). Training augmentation (gain, noise,
speed, reverb, SpecAugment) lives here too, so the model side never sees
raw audio.

In [ ]:
import math
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [ ]:
SAMPLE_RATE = 16000
N_FFT = 400
HOP_LENGTH = 160
CHUNK_LENGTH = 30
N_SAMPLES = CHUNK_LENGTH * SAMPLE_RATE
N_FRAMES = N_SAMPLES // HOP_LENGTH
N_MELS = 80
FMIN = 0.0
FMAX = 8000.0

In [ ]:
def hz_to_mel(f):
    f = np.asarray(f, dtype=np.float64)
    min_log_hz = 1000.0
    min_log_mel = 15.0
    logstep = 27.0 / np.log(6.4)
    mel = 3.0 * f / 200.0
    log_region = f >= min_log_hz
    mel = np.where(log_region, min_log_mel + np.log(np.maximum(f, min_log_hz) / min_log_hz) * logstep, mel)
    return mel

def mel_to_hz(m):
    m = np.asarray(m, dtype=np.float64)
    min_log_hz = 1000.0
    min_log_mel = 15.0
    logstep = np.log(6.4) / 27.0
    f = 200.0 * m / 3.0
    log_region = m >= min_log_mel
    f = np.where(log_region, min_log_hz * np.exp(logstep * (np.maximum(m, min_log_mel) - min_log_mel)), f)
    return f

def mel_filterbank(sr=SAMPLE_RATE, n_fft=N_FFT, n_mels=N_MELS, fmin=FMIN, fmax=FMAX):
    fft_freqs = np.linspace(0, sr / 2, n_fft // 2 + 1)
    mel_pts = np.linspace(hz_to_mel(fmin), hz_to_mel(fmax), n_mels + 2)
    hz_pts = mel_to_hz(mel_pts)
    fb = np.zeros((n_mels, len(fft_freqs)), dtype=np.float64)
    fdiff = np.diff(hz_pts)
    ramps = hz_pts[:, None] - fft_freqs[None, :]
    for i in range(n_mels):
        lower = -ramps[i] / fdiff[i]
        upper = ramps[i + 2] / fdiff[i + 1]
        fb[i] = np.maximum(0, np.minimum(lower, upper))
    enorm = 2.0 / (hz_pts[2:n_mels + 2] - hz_pts[:n_mels])
    fb *= enorm[:, None]
    return fb.astype(np.float32)

FILTERS = torch.from_numpy(mel_filterbank())
print(FILTERS.shape, float(FILTERS.sum()))

In [ ]:
np.savez_compressed(
    "assets/mel_filters.npz",
    mel_80=mel_filterbank(n_mels=80),
    mel_128=mel_filterbank(n_mels=128),
)

In [ ]:
def hann_window(n=N_FFT):
    return torch.hann_window(n, periodic=True)

def stft_magnitudes(audio):
    window = hann_window().to(audio.device)
    stft = torch.stft(
        audio,
        N_FFT,
        HOP_LENGTH,
        window=window,
        center=True,
        pad_mode="reflect",
        return_complex=True,
    )
    return stft[..., :-1].abs() ** 2

def log_mel_spectrogram(audio, n_mels=N_MELS, padding=0):
    if not torch.is_tensor(audio):
        audio = torch.from_numpy(np.asarray(audio, dtype=np.float32))
    if padding > 0:
        audio = F.pad(audio, (0, padding))
    magnitudes = stft_magnitudes(audio)
    filters = FILTERS.to(audio.device) if n_mels == 80 else torch.from_numpy(mel_filterbank(n_mels=n_mels)).to(audio.device)
    mel_spec = filters @ magnitudes
    log_spec = torch.clamp(mel_spec, min=1e-10).log10()
    log_spec = torch.maximum(log_spec, log_spec.max() - 8.0)
    log_spec = (log_spec + 4.0) / 4.0
    return log_spec

def pad_or_trim(audio, length=N_SAMPLES):
    if audio.shape[-1] > length:
        return audio[..., :length]
    if audio.shape[-1] < length:
        return F.pad(audio, (0, length - audio.shape[-1]))
    return audio

In [ ]:
t = torch.linspace(0, 3.0, int(3.0 * SAMPLE_RATE))
tone = 0.4 * torch.sin(2 * math.pi * 440.0 * t) + 0.2 * torch.sin(2 * math.pi * 1320.0 * t)
mel = log_mel_spectrogram(pad_or_trim(tone))
print(mel.shape, float(mel.min()), float(mel.max()))

fig, ax = plt.subplots(figsize=(12, 3))
ax.imshow(mel.numpy(), aspect="auto", origin="lower")
ax.set_xlabel("frame")
ax.set_ylabel("mel bin")
fig.tight_layout()

In [ ]:
def sweep(f0, f1, seconds=5.0):
    t = torch.linspace(0, seconds, int(seconds * SAMPLE_RATE))
    phase = 2 * math.pi * (f0 * t + (f1 - f0) * t ** 2 / (2 * seconds))
    return 0.5 * torch.sin(phase)

mel_sweep = log_mel_spectrogram(pad_or_trim(sweep(80.0, 7800.0)))
fig, ax = plt.subplots(figsize=(12, 3))
ax.imshow(mel_sweep.numpy(), aspect="auto", origin="lower")
fig.tight_layout()

In [ ]:
class GainAugment:
    def __init__(self, low_db=-12.0, high_db=6.0):
        self.low = low_db
        self.high = high_db

    def __call__(self, wave, rng):
        db = rng.uniform(self.low, self.high)
        return wave * (10.0 ** (db / 20.0))

class NoiseAugment:
    def __init__(self, snr_low_db=5.0, snr_high_db=30.0):
        self.low = snr_low_db
        self.high = snr_high_db

    def __call__(self, wave, rng):
        snr = rng.uniform(self.low, self.high)
        signal_power = float((wave ** 2).mean()) + 1e-12
        noise_power = signal_power / (10.0 ** (snr / 10.0))
        noise = torch.randn_like(wave) * math.sqrt(noise_power)
        return wave + noise

class SpeedAugment:
    def __init__(self, low=0.9, high=1.1):
        self.low = low
        self.high = high

    def __call__(self, wave, rng):
        factor = rng.uniform(self.low, self.high)
        n_out = int(len(wave) / factor)
        idx = torch.linspace(0, len(wave) - 1, n_out)
        lo = idx.floor().long()
        hi = torch.clamp(lo + 1, max=len(wave) - 1)
        frac = idx - lo.float()
        return wave[lo] * (1 - frac) + wave[hi] * frac

class ReverbAugment:
    def __init__(self, rt60_low=0.1, rt60_high=0.6):
        self.low = rt60_low
        self.high = rt60_high

    def __call__(self, wave, rng):
        rt60 = rng.uniform(self.low, self.high)
        n = int(rt60 * SAMPLE_RATE)
        decay = torch.exp(-6.908 * torch.arange(n) / n)
        impulse = torch.randn(n) * decay
        impulse[0] = 1.0
        impulse = impulse / impulse.norm()
        out = F.conv1d(wave.view(1, 1, -1), impulse.flip(0).view(1, 1, -1), padding=n - 1)
        return out.view(-1)[: len(wave)]

In [ ]:
class SpecAugment:
    def __init__(self, freq_masks=2, freq_width=27, time_masks=2, time_width_ratio=0.05):
        self.freq_masks = freq_masks
        self.freq_width = freq_width
        self.time_masks = time_masks
        self.time_width_ratio = time_width_ratio

    def __call__(self, mel, rng):
        mel = mel.clone()
        n_mels, n_frames = mel.shape
        fill = float(mel.mean())
        for _ in range(self.freq_masks):
            w = rng.randint(0, self.freq_width)
            f0 = rng.randint(0, max(n_mels - w, 1))
            mel[f0:f0 + w, :] = fill
        max_t = int(n_frames * self.time_width_ratio)
        for _ in range(self.time_masks):
            w = rng.randint(0, max(max_t, 1))
            t0 = rng.randint(0, max(n_frames - w, 1))
            mel[:, t0:t0 + w] = fill
        return mel

import random
rng = random.Random(7)
aug = SpecAugment()
masked = aug(mel_sweep, rng)
fig, ax = plt.subplots(figsize=(12, 3))
ax.imshow(masked.numpy(), aspect="auto", origin="lower")
fig.tight_layout()

In [ ]:
class FrontendPipeline:
    def __init__(self, train=True, seed=0):
        self.train = train
        self.rng = random.Random(seed)
        self.wave_augs = [GainAugment(), NoiseAugment(), SpeedAugment()]
        self.reverb = ReverbAugment()
        self.spec_aug = SpecAugment()

    def __call__(self, wave):
        if self.train:
            for a in self.wave_augs:
                if self.rng.random() < 0.5:
                    wave = a(wave, self.rng)
            if self.rng.random() < 0.15:
                wave = self.reverb(wave, self.rng)
        wave = pad_or_trim(wave)
        mel = log_mel_spectrogram(wave)
        if self.train and self.rng.random() < 0.8:
            mel = self.spec_aug(mel, self.rng)
        return mel

pipe = FrontendPipeline(train=True, seed=11)
out = pipe(sweep(120.0, 4000.0, seconds=8.0))
print(out.shape)

In [ ]:
def frontend_parity_vectors():
    cases = {}
    torch.manual_seed(3)
    for name, wave in {
        "silence": torch.zeros(SAMPLE_RATE),
        "tone_440": 0.4 * torch.sin(2 * math.pi * 440.0 * torch.linspace(0, 1, SAMPLE_RATE)),
        "noise": torch.randn(SAMPLE_RATE) * 0.1,
    }.items():
        mel = log_mel_spectrogram(pad_or_trim(wave))
        cases[name] = {
            "shape": list(mel.shape),
            "mean": float(mel.mean()),
            "std": float(mel.std()),
            "first_frame": [round(float(x), 6) for x in mel[:, 0][:8]],
        }
    return cases

Path("assets").mkdir(exist_ok=True)
with open("assets/frontend-parity.json", "w") as f:
    json.dump(frontend_parity_vectors(), f, indent=2)
print(json.dumps(frontend_parity_vectors(), indent=2))